### Rain-on-snow project <br>
#### USGS streamflow time series retrieval

This notebook retrieves NWM retrospective streamflow and USGS observed streaflow for GAGES II (reference and non-reference basins).<br>
Streamflow time series are retrieved using USGS Water dat API, Dataretrieval package. See documentation in: <br>
https://doi-usgs.github.io/dataretrieval-python/index.html <br>
https://github.com/DOI-USGS/dataretrieval-python/tree/main <br>

To have access to higher rate limits, an API Key can be requested at:<br>
https://api.waterdata.usgs.gov/signup/

Once time series are retrieved, several performance metrics will be computed, including:
- NSE
- KGE
- PBias
- Pearson Correlation

In [1]:
import sys
# Path to ROS functions
sys.path.append('/users/m/m/mmorale3/netfiles/ciroh/mmorales/ROS_project/code_testing/ros-workflow')

In [2]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime
import numpy as np
from dataretrieval import waterdata
import functions.rain_on_snow_fncns_mirror as ros

1. User specific definitions

In [3]:
######### local paths
# base directory where input metadata is located
base_dir = Path('./input')

# output directory for retrieved data
output_dir = Path('./output')

# location list (USGS gauge IDs)
locations_path = Path(base_dir, 'Metadata_GAGESII_NE_ROS.parquet')

# unique ID column header in the locations list
unique_location_id = 'GAGE_ID'

2. Read locations and retrieve streamflow time series

In [4]:
# create a list of usgs IDs in the format "usgs-xxxxxxxx"
df_ids = pd.read_parquet(locations_path, engine='pyarrow')
id_list = df_ids[unique_location_id].radd('USGS-').to_list()
print(f'Total of GAGES II sites: {len(id_list)}')

Total of GAGES II sites: 913


In [ ]:
# Setting API Key as environment variable to allow higher rate limits
os.environ["API_USGS_PAT"] = "QurcJrqNEtcAdf4hrp63e0MYe1ZrjDCaqFodaMud"

In [ ]:
# Example for retrieving few sites
siteIDs = id_list[0:2]
parameterCode = "00060"  # Discharge
startDate = "2021-09-01" #"1979-10-01" #
endDate = "2021-09-30" #"2022-09-30" #

# Get the data
discharge = waterdata.get_continuous(
    monitoring_location_id=siteIDs, parameter_code=parameterCode, time=f"{startDate}/{endDate}"
)
print("Retrieved " + str(len(discharge[0])) + " data values.")

* Retrieving all sites across the NE

In [6]:
%%time
# Set the parameters for the USGS data retrieval
parameterCode = '00060'  # Discharge
startDate = '1979-10-01'
endDate   = '2022-09-30'
batchSize  = 40    # sites per request; lower to 20-25 if timeouts occur
nWorkers   = 5   # concurrent HTTP requests; lower if the NWIS server returns errors
usgs_output_path = output_dir / 'usgs_streamflow_Q.parquet'

# Intermediate chunk files are saved to output/usgs_streamflow_Q_chunks/ as each
# request completes. If the run is interrupted, re-running this cell will skip
# already-saved chunks and resume from where it left off.
ros.fetch_usgs_data(
    site_list=id_list,
    start_date=startDate,
    end_date=endDate,
    parameter_code=parameterCode,
    batch_size=batchSize,
    max_workers=nWorkers,
    output_path=usgs_output_path
)

Total tasks: 368 | Already done: 368 | Submitting: 0 | Workers: 5

Merging 368 chunk files (streaming, normalized schema)...
Merged → output/usgs_streamflow_Q.parquet (590,356,537 rows (44 empty skipped))
Done! All chunks complete.
CPU times: user 7min 47s, sys: 1min 4s, total: 8min 52s
Wall time: 7min 43s


In [ ]:
# Display the data frame as a table
display(discharge[0])

In [ ]:
print(discharge[0].dtypes)